# Movie Ratings: Normalization vs. Standardization
By: Shehou Fisiru


In [13]:
import pandas as pd

ratings = pd.read_csv("ratings.csv", index_col="Name")
ratings

,Avengers: Endgame,Spider-Man: Brand New Day,Captain America: Civil War,X-Men: Days of Future Past,Logan,Fantastic Four: First Steps
Name,,,,,,
Elisha,5,5.0,4,4.0,5.0,4
Wilson,3,NaN,3,2.0,4.0,1
Sebastian,4,NaN,5,3.0,4.0,3
Mikey,5,4.0,4,NaN,NaN,3
Kai,4,NaN,3,4.0,3.0,3


## Average ratings

In [14]:
def show_averages(df, label):
    user_avg = df.mean(axis=1).round(2).replace(-0.0, 0.0)
    movie_avg = pd.DataFrame({
        "avg_rating": df.mean(axis=0).round(2).replace(-0.0, 0.0),
        "num_ratings": df.count(axis=0),
    }).sort_values("avg_rating", ascending=False)

    print(f"Average {label} rating by user:")
    print(user_avg.sort_values(ascending=False).to_string())
    print(f"\nAverage {label} rating by movie:")
    print(movie_avg.to_string())

show_averages(ratings, "raw")

Average raw rating by user:
Name
Elisha       4.5
Mikey        4.0
Sebastian    3.8
Kai          3.4
Wilson       2.6

Average raw rating by movie:
                             avg_rating  num_ratings
Spider-Man: Brand New Day          4.50            2
Avengers: Endgame                  4.20            5
Logan                              4.00            4
Captain America: Civil War         3.80            5
X-Men: Days of Future Past         3.25            4
Fantastic Four: First Steps        2.80            5


## Normalized ratings

I normalized the data, so each person's lowest rating becomes 0 and highest becomes 1.

In [15]:
user_min = ratings.min(axis=1)
user_max = ratings.max(axis=1)

normalized = ratings.sub(user_min, axis=0).div(user_max - user_min, axis=0)
normalized.round(2)

,Avengers: Endgame,Spider-Man: Brand New Day,Captain America: Civil War,X-Men: Days of Future Past,Logan,Fantastic Four: First Steps
Name,,,,,,
Elisha,1.00,1.0,0.00,0.00,1.0,0.0
Wilson,0.67,NaN,0.67,0.33,1.0,0.0
Sebastian,0.50,NaN,1.00,0.00,0.5,0.0
Mikey,1.00,0.5,0.50,NaN,NaN,0.0
Kai,1.00,NaN,0.00,1.00,0.0,0.0


In [16]:
show_averages(normalized, "normalized")

Average normalized rating by user:
Name
Wilson       0.53
Elisha       0.50
Mikey        0.50
Sebastian    0.40
Kai          0.40

Average normalized rating by movie:
                             avg_rating  num_ratings
Avengers: Endgame                  0.83            5
Spider-Man: Brand New Day          0.75            2
Logan                              0.62            4
Captain America: Civil War         0.43            5
X-Men: Days of Future Past         0.33            4
Fantastic Four: First Steps        0.00            5


## Conclusion

Normalizing each user’s ratings removes personal rating bias. For example, since Elisha never rates below a 4 while Wilson gave a 1, each rating instead reflects how a movie ranks relative to that person’s own taste. This shifted the results as Avengers: Endgame rose to the top, Fantastic Four: First Steps dropped to 0.00 as everyone’s lowest, and Spider-Man: Brand New Day lost its raw lead, which only came from its two raters. However, normalization also exaggerates small differences, turning Kai’s 3s and 4s into 0s and 1s, and it loses absolute meaning, since Elisha’s 0 for Fantastic Four: First Steps was still a 4 out of 5. It’s also fragile with small samples, because each user’s scale depends on just their highest and lowest rating. Overall, raw ratings are better for judging whether a movie is good, while normalized ratings are better for comparing preferences across people.

## Extra credit: standardized ratings

Z-scores per user: 0 is that person's average, positive is above it, negative is below.

In [17]:
user_mean = ratings.mean(axis=1)
user_std = ratings.std(axis=1)

standardized = ratings.sub(user_mean, axis=0).div(user_std, axis=0)
standardized.round(2)

,Avengers: Endgame,Spider-Man: Brand New Day,Captain America: Civil War,X-Men: Days of Future Past,Logan,Fantastic Four: First Steps
Name,,,,,,
Elisha,0.91,0.91,-0.91,-0.91,0.91,-0.91
Wilson,0.35,NaN,0.35,-0.53,1.23,-1.40
Sebastian,0.24,NaN,1.43,-0.96,0.24,-0.96
Mikey,1.22,0.00,0.00,NaN,NaN,-1.22
Kai,1.10,NaN,-0.73,1.10,-0.73,-0.73


In [18]:
show_averages(standardized, "standardized")

Average standardized rating by user:
Name
Elisha       0.0
Wilson       0.0
Sebastian    0.0
Mikey        0.0
Kai          0.0

Average standardized rating by movie:
                             avg_rating  num_ratings
Avengers: Endgame                  0.76            5
Spider-Man: Brand New Day          0.46            2
Logan                              0.41            4
Captain America: Civil War         0.03            5
X-Men: Days of Future Past        -0.32            4
Fantastic Four: First Steps       -1.05            5


User averages are all 0 by definition. The movie ranking matches the normalized one, with Endgame on top and FF: First Steps at the bottom.